# SageMaker Pipeline: AutoGluon Tabular Classification

End-to-end ML pipeline using SageMaker SDK v3:
1. **PreprocessData** — ScriptProcessor with sklearn to split raw data
2. **TrainAutoGluon** — ModelTrainer with AutoGluon DLC
3. **EvaluateModel** — Compute metrics on test set

## Configuration

In [ ]:
import boto3
import sagemaker

REGION = boto3.session.Session().region_name
sess = sagemaker.session.Session()
BUCKET = sess.default_bucket()
S3_PREFIX = "autogluon-tabular"

AG_VERSION = "1.5"
PY_VERSION = "py312"
PIPELINE_NAME = "AutoGluonTabularPipeline"

## Discover IAM role

In [ ]:
def get_role(role_arn=None):
    if role_arn:
        return role_arn
    iam = boto3.client("iam")
    paginator = iam.get_paginator("list_roles")
    for page in paginator.paginate():
        for role in page["Roles"]:
            if "SageMaker" in role["RoleName"] or "sagemaker" in role["RoleName"]:
                return role["Arn"]
    raise ValueError("No SageMaker IAM role found. Pass role_arn explicitly.")

role_arn = get_role()
print(f"Role: {role_arn}")

## Upload config.yaml to S3

The training step needs the config file available in S3.

In [ ]:
import os

s3 = boto3.client("s3", region_name=REGION)
config_key = f"{S3_PREFIX}/pipeline/config/config.yaml"
config_path = os.path.abspath(os.path.join("..", "1-training", "config.yaml"))
s3.upload_file(config_path, BUCKET, config_key)
print(f"Uploaded config to s3://{BUCKET}/{config_key}")

## Define the pipeline

All pipeline steps are defined inside `create_pipeline()`.

In [ ]:
from sagemaker.core import image_uris
from sagemaker.core.processing import ProcessingInput, ProcessingOutput, ScriptProcessor
from sagemaker.core.shapes.shapes import ProcessingS3Input, ProcessingS3Output
from sagemaker.core.training.configs import (
    Compute, OutputDataConfig, SourceCode, StoppingCondition,
)
from sagemaker.core.workflow.parameters import ParameterString
from sagemaker.core.workflow.pipeline_context import PipelineSession
from sagemaker.core.workflow.properties import PropertyFile
from sagemaker.mlops.workflow.pipeline import Pipeline
from sagemaker.mlops.workflow.steps import ProcessingStep, TrainingStep
from sagemaker.train import ModelTrainer


def create_pipeline(role_arn, region, bucket, ag_version="1.5", py_version="py312",
                    pipeline_name="AutoGluonTabularPipeline"):
    pipeline_session = PipelineSession()

    # Pipeline Parameters
    input_data_uri = ParameterString(name="InputDataUri", default_value=f"s3://{bucket}/autogluon-tabular/raw/")
    instance_type = ParameterString(name="InstanceType", default_value="ml.m5.2xlarge")
    s3_prefix = f"s3://{bucket}/autogluon-tabular/pipeline"

    # Image URIs
    training_image = image_uris.retrieve("autogluon", region=region, version=ag_version,
        py_version=py_version, image_scope="training", instance_type="ml.m5.2xlarge")
    processing_image = image_uris.retrieve("sklearn", region=region, version="1.2-1")

    # Step 1: Preprocess — use step_args=processor.run(...) pattern
    preprocessor = ScriptProcessor(image_uri=processing_image, role=role_arn, command=["python3"],
        instance_type="ml.m5.xlarge", instance_count=1, sagemaker_session=pipeline_session)

    step_preprocess = ProcessingStep(name="PreprocessData", step_args=preprocessor.run(
        code=os.path.abspath(os.path.join("..", "0-data-prep", "preprocess.py")),
        inputs=[ProcessingInput(input_name="input", s3_input=ProcessingS3Input(
            s3_uri=input_data_uri, local_path="/opt/ml/processing/input", s3_data_type="S3Prefix"))],
        outputs=[
            ProcessingOutput(output_name="train", s3_output=ProcessingS3Output(
                s3_uri=f"{s3_prefix}/processed/train/", local_path="/opt/ml/processing/train", s3_upload_mode="EndOfJob")),
            ProcessingOutput(output_name="test", s3_output=ProcessingS3Output(
                s3_uri=f"{s3_prefix}/processed/test/", local_path="/opt/ml/processing/test", s3_upload_mode="EndOfJob")),
        ],
    ))

    # Step 2: Train
    trainer = ModelTrainer(training_image=training_image, role=role_arn,
        source_code=SourceCode(source_dir=os.path.abspath(os.path.join("..", "1-training")), entry_script="train.py"),
        compute=Compute(instance_type=instance_type, instance_count=1, volume_size_in_gb=100, keep_alive_period_in_seconds=0),
        output_data_config=OutputDataConfig(s3_output_path=f"{s3_prefix}/model/"),
        base_job_name="ag-tabular-train",
        stopping_condition=StoppingCondition(max_runtime_in_seconds=7200),
        sagemaker_session=pipeline_session)

    step_train = TrainingStep(name="TrainAutoGluon", step_args=trainer.train(input_data_config=[
        {"channel_name": "train", "data_source": {"s3_data_source": {"s3_uri": step_preprocess.properties.ProcessingOutputConfig.Outputs["train"].S3Output.S3Uri, "s3_data_type": "S3Prefix"}}},
        {"channel_name": "test", "data_source": {"s3_data_source": {"s3_uri": step_preprocess.properties.ProcessingOutputConfig.Outputs["test"].S3Output.S3Uri, "s3_data_type": "S3Prefix"}}},
        {"channel_name": "config", "data_source": {"s3_data_source": {"s3_uri": f"{s3_prefix}/config/", "s3_data_type": "S3Prefix"}}},
    ]))

    # Step 3: Evaluate
    evaluation_report = PropertyFile(name="EvaluationReport", output_name="evaluation", path="evaluation.json")
    evaluator = ScriptProcessor(image_uri=training_image, role=role_arn, command=["python3"],
        instance_type="ml.m5.xlarge", instance_count=1, sagemaker_session=pipeline_session)

    step_evaluate = ProcessingStep(name="EvaluateModel", step_args=evaluator.run(
        code=os.path.abspath("evaluate.py"),
        inputs=[
            ProcessingInput(input_name="model", s3_input=ProcessingS3Input(
                s3_uri=step_train.properties.ModelArtifacts.S3ModelArtifacts,
                local_path="/opt/ml/processing/model", s3_data_type="S3Prefix")),
            ProcessingInput(input_name="test", s3_input=ProcessingS3Input(
                s3_uri=step_preprocess.properties.ProcessingOutputConfig.Outputs["test"].S3Output.S3Uri,
                local_path="/opt/ml/processing/test", s3_data_type="S3Prefix")),
        ],
        outputs=[ProcessingOutput(output_name="evaluation", s3_output=ProcessingS3Output(
            s3_uri=f"{s3_prefix}/evaluation/", local_path="/opt/ml/processing/evaluation", s3_upload_mode="EndOfJob"))],
    ), property_files=[evaluation_report])

    # Assemble Pipeline
    pipeline = Pipeline(name=pipeline_name,
        parameters=[input_data_uri, instance_type],
        steps=[step_preprocess, step_train, step_evaluate],
        sagemaker_session=pipeline_session)

    return pipeline

## Create and upsert the pipeline

In [ ]:
pipeline = create_pipeline(
    role_arn=role_arn,
    region=REGION,
    bucket=BUCKET,
    ag_version=AG_VERSION,
    py_version=PY_VERSION,
    pipeline_name=PIPELINE_NAME,
)

pipeline.upsert(role_arn=role_arn)
print(f"Pipeline '{PIPELINE_NAME}' created/updated.")

## Execute the pipeline

Uncomment and run the cell below to start a pipeline execution.

In [ ]:
# execution = pipeline.start()
# print(f"Pipeline execution started: {execution.describe()['PipelineExecutionArn']}")
# execution.wait()
# print("Pipeline execution complete.")
# print(execution.describe())